# 02 · Smoke test: Naive, MA(5), ARIMA

Same models as the midterm, now run through the **shared walk-forward code** (`src/finx_common.py`) on the frozen master `data/gold/master.csv`. Two checks:
1. RMSE matches the midterm (Naive 0.966, MA(5) 1.073, ARIMA 0.964; difference > 0.01 = bug).
2. Training time and latency (ms per day) print for every model.

*Edited from the midterm baselines notebook (kept in `notebooks/archive_midterm/`).*

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
import config as C, baselines as B, make_results
from finx_common import load_master, walk_forward
OUT = str(C.OUT_DIR)          # outputs/
df = load_master()            # data/gold/master.csv (frozen master)
df.shape

(1435, 59)

In [2]:
for name, fit, pred in [("naive", B.naive_fit, B.naive_predict), ("ma5", B.ma_fit, B.ma_predict), ("arima", B.arima_fit, B.arima_predict)]:
    walk_forward(fit, pred, name, df=df, out_dir=OUT)

[naive] fold 2024: train 753 days, test 252 days, train 0.0s, 0.041 ms for one prediction
[naive] fold 2025: train 1005 days, test 250 days, train 0.0s, 0.039 ms for one prediction
[naive] fold 2026: train 1255 days, test 180 days, train 0.0s, 0.039 ms for one prediction
[ma5] fold 2024: train 753 days, test 252 days, train 0.0s, 0.125 ms for one prediction
[ma5] fold 2025: train 1005 days, test 250 days, train 0.0s, 0.122 ms for one prediction
[ma5] fold 2026: train 1255 days, test 180 days, train 0.0s, 0.130 ms for one prediction


[arima] fold 2024: train 753 days, test 252 days, train 1.3s, 2.217 ms for one prediction


[arima] fold 2025: train 1005 days, test 250 days, train 1.5s, 2.604 ms for one prediction


[arima] fold 2026: train 1255 days, test 180 days, train 1.8s, 3.297 ms for one prediction


## Results (all test days pooled; RMSE and MAE in % points)

In [3]:
tabs = make_results.main(OUT)
tabs['results_table.csv']

                          Model      Tier  RMSE (%)  MAE (%)  Direction (%)  Skill vs naive (%)  Train time per fold (s)  Inference per day (ms)
             Naive (return = 0)  Baseline     0.966    0.658            NaN               0.000                    0.000                   0.038
         Moving average (5-day)  Baseline     1.073    0.729         50.147             -11.028                    0.000                   0.127
                          ARIMA  Baseline     0.964    0.655         54.106               0.228                    1.541                   2.706
Always up (direction benchmark) Benchmark       NaN      NaN         56.012                 NaN                      NaN                     NaN
Hardware: {'cpu': 'arm', 'cores': 11, 'gpu': 'none'}


,Model,Tier,RMSE (%),MAE (%),Direction (%),Skill vs naive (%),Train time per fold (s),Inference per day (ms)
0,Naive (return = 0),Baseline,0.966119,0.657798,NaN,0.000000,2.499825e-07,0.038334
1,Moving average (5-day),Baseline,1.072661,0.728966,50.146628,-11.027885,2.633703e-07,0.127007
2,ARIMA,Baseline,0.963918,0.654726,54.105572,0.227770,1.541294e+00,2.705840
3,Always up (direction benchmark),Benchmark,NaN,NaN,56.011730,NaN,NaN,NaN


## Check 1: compare with the midterm

In [4]:
MIDTERM = {"Naive (return = 0)": 0.966, "Moving average (5-day)": 1.073, "ARIMA": 0.964}
t = tabs["results_table.csv"].set_index("Model")
{m: (round(float(t.loc[m, "RMSE (%)"]), 3), v, abs(t.loc[m, "RMSE (%)"] - v) <= 0.01) for m, v in MIDTERM.items()}
# (now, midterm, within 0.01?)  -- the full pass/fail check is `python src/smoke_test.py`.

{'Naive (return = 0)': (0.966, 0.966, np.True_),
 'Moving average (5-day)': (1.073, 1.073, np.True_),
 'ARIMA': (0.964, 0.964, np.True_)}

## Check 2: timers
Train time and latency must be filled in for every model (see table above).

In [5]:
assert t.loc["ARIMA", "Train time per fold (s)"] > 0 and (t["Inference per day (ms)"].dropna() > 0).all()
print("Timers OK")

Timers OK


## Per fold (Table A) and without the 5 biggest-move days (Table B)

In [6]:
tabs['table_A_by_fold.csv']

,Model,RMSE 2024,Direction 2024,RMSE 2025,Direction 2025,RMSE 2026,Direction 2026
0,Naive (return = 0),0.799281,NaN,1.180857,NaN,0.839883,NaN
1,Moving average (5-day),0.889239,52.380952,1.325731,50.4,0.900777,46.666667
2,ARIMA,0.796047,56.746032,1.178989,58.0,0.838248,45.000000


In [7]:
tabs['table_B_no_big_days.csv']

,Model,RMSE all days,RMSE without 5 biggest days,Rank change
0,Naive (return = 0),0.966119,0.828167,0
1,Moving average (5-day),1.072661,0.923429,0
2,ARIMA,0.963918,0.825435,0


In [8]:
tabs['calm_vs_stressed.csv']

,Model,RMSE calm,Direction calm,RMSE stressed,Direction stressed
0,Naive (return = 0),0.729899,NaN,1.643551,NaN
1,Moving average (5-day),0.791590,50.719424,1.860860,47.619048
2,ARIMA,0.731537,52.517986,1.633311,61.111111
